In [ ]:
!pip install -r ../requirements.txt

In [1]:
import os
import re
import math
import random
import warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
np.random.seed(42)
random.seed(42)

In [2]:
# Spark session
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType, StructField, IntegerType, FloatType, LongType, StringType
)
from pyspark.sql import functions as F
from pyspark.storagelevel import StorageLevel

spark = SparkSession.builder \
    .appName("Assignment03_MovieLens_Recommender") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/15 07:59:33 WARN Utils: Your hostname, ASHISHs-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.5 instead (on interface en0)
26/03/15 07:59:33 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/15 07:59:33 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 4.1.1


In [3]:
# Dataset paths
project_root = Path.cwd().resolve().parent
data_dir = project_root / "dataset" / "ml-latest-small"

ratings_file = (data_dir / "ratings.csv").resolve()
movies_file = (data_dir / "movies.csv").resolve()
tags_file = (data_dir / "tags.csv").resolve()

assert ratings_file.exists(), f"ratings.csv not found: {ratings_file}"
assert movies_file.exists(), f"movies.csv not found: {movies_file}"

ratings_path = f"file://{ratings_file}"
movies_path = f"file://{movies_file}"
tags_path = f"file://{tags_file}" if tags_file.exists() else None

print("Ratings path:", ratings_path)
print("Movies path :", movies_path)

Ratings path: file:///Users/ashishsinha/Documents/Study/MTech/Sm02/MLwDE/Assignment03/gihubrepo/iitj_m25de1047_mlbd_Assignment03/dataset/ml-latest-small/ratings.csv
Movies path : file:///Users/ashishsinha/Documents/Study/MTech/Sm02/MLwDE/Assignment03/gihubrepo/iitj_m25de1047_mlbd_Assignment03/dataset/ml-latest-small/movies.csv


In [4]:
# Explicit schemas and load data
ratings_schema = StructType([
    StructField("userId", IntegerType(), True),
    StructField("movieId", IntegerType(), True),
    StructField("rating", FloatType(), True),
    StructField("timestamp", LongType(), True)
])

movies_schema = StructType([
    StructField("movieId", IntegerType(), True),
    StructField("title", StringType(), True),
    StructField("genres", StringType(), True)
])

ratings = spark.read.csv(ratings_path, header=True, schema=ratings_schema)
movies = spark.read.csv(movies_path, header=True, schema=movies_schema)

ratings = ratings.persist(StorageLevel.MEMORY_AND_DISK)
movies = movies.persist(StorageLevel.MEMORY_AND_DISK)

print("Ratings count:", ratings.count())
print("Movies count :", movies.count())

ratings.show(5, truncate=False)
movies.show(5, truncate=False)
ratings.printSchema()
movies.printSchema()

Ratings count: 100836
Movies count : 9742
+------+-------+------+---------+
|userId|movieId|rating|timestamp|
+------+-------+------+---------+
|1     |1      |4.0   |964982703|
|1     |3      |4.0   |964981247|
|1     |6      |4.0   |964982224|
|1     |47     |5.0   |964983815|
|1     |50     |5.0   |964982931|
+------+-------+------+---------+
only showing top 5 rows
+-------+----------------------------------+-------------------------------------------+
|movieId|title                             |genres                                     |
+-------+----------------------------------+-------------------------------------------+
|1      |Toy Story (1995)                  |Adventure|Animation|Children|Comedy|Fantasy|
|2      |Jumanji (1995)                    |Adventure|Children|Fantasy                 |
|3      |Grumpier Old Men (1995)           |Comedy|Romance                             |
|4      |Waiting to Exhale (1995)          |Comedy|Drama|Romance                       |
|5   

In [5]:
# Preprocessing
movies = movies.withColumn(
    "genres_clean",
    F.when(
        F.col("genres").isNull() | (F.col("genres") == "(no genres listed)"),
        F.lit("Unknown")
    ).otherwise(F.col("genres"))
)

year_str = F.regexp_extract(F.col("title"), r"\((\d{4})\)", 1)

movies = movies.withColumn(
    "year",
    F.when(year_str != "", year_str.cast("int")).otherwise(F.lit(None).cast("int"))
)

movie_stats = ratings.groupBy("movieId").agg(
    F.avg("rating").alias("movie_avg_rating"),
    F.count("*").alias("movie_rating_count")
)

user_stats = ratings.groupBy("userId").agg(
    F.avg("rating").alias("user_avg_rating"),
    F.count("*").alias("user_rating_count")
)

ratings_enriched = ratings.join(movie_stats, "movieId", "left").join(user_stats, "userId", "left")
ratings_enriched.show(5, truncate=False)

+------+-------+------+---------+------------------+------------------+-----------------+-----------------+
|userId|movieId|rating|timestamp|movie_avg_rating  |movie_rating_count|user_avg_rating  |user_rating_count|
+------+-------+------+---------+------------------+------------------+-----------------+-----------------+
|1     |1      |4.0   |964982703|3.9209302325581397|215               |4.366379310344827|232              |
|1     |3      |4.0   |964981247|3.2596153846153846|52                |4.366379310344827|232              |
|1     |6      |4.0   |964982224|3.946078431372549 |102               |4.366379310344827|232              |
|1     |47     |5.0   |964983815|3.9753694581280787|203               |4.366379310344827|232              |
|1     |50     |5.0   |964982931|4.237745098039215 |204               |4.366379310344827|232              |
+------+-------+------+---------+------------------+------------------+-----------------+-----------------+
only showing top 5 rows


In [6]:
# Train-test split
split_ts = ratings.approxQuantile("timestamp", [0.8], 0.01)[0]

train_ratings = ratings.filter(F.col("timestamp") <= split_ts).persist(StorageLevel.MEMORY_AND_DISK)
test_ratings = ratings.filter(F.col("timestamp") > split_ts).persist(StorageLevel.MEMORY_AND_DISK)

print("Train ratings:", train_ratings.count())
print("Test ratings :", test_ratings.count())

Train ratings: 80086
Test ratings : 20750


In [7]:
# Helper fucntions
from sklearn.metrics import mean_squared_error
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse.linalg import svds

TOP_K = 5
POSITIVE_THRESHOLD = 4.0

def rmse_score(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def clip_ratings(arr, low=0.5, high=5.0):
    return np.clip(arr, low, high)

def precision_recall_at_k(pred_df, truth_df, k=5, positive_threshold=4.0):
    truth_pos = truth_df[truth_df["rating"] >= positive_threshold].groupby("userId")["movieId"].apply(set).to_dict()

    pred_topk = (
        pred_df.sort_values(["userId", "score"], ascending=[True, False])
               .groupby("userId")
               .head(k)
               .groupby("userId")["movieId"]
               .apply(list)
               .to_dict()
    )

    precisions, recalls = [], []
    for user, recs in pred_topk.items():
        truth = truth_pos.get(user, set())
        if not truth:
            continue
        hits = len(set(recs) & truth)
        precisions.append(hits / k)
        recalls.append(hits / len(truth))
    return float(np.mean(precisions)) if precisions else 0.0, float(np.mean(recalls)) if recalls else 0.0

# Part 1 — Content-Based Filtering

## Task 1:

In [8]:
# TF-IDF on genres
movies_pd = movies.select("movieId", "title", "genres_clean", "year").toPandas().copy()
movies_pd["genres_clean"] = movies_pd["genres_clean"].fillna("Unknown")

tfidf = TfidfVectorizer(token_pattern=r"[^|]+")
tfidf_matrix = tfidf.fit_transform(movies_pd["genres_clean"])

print("TF-IDF matrix shape:", tfidf_matrix.shape)

TF-IDF matrix shape: (9742, 20)


In [9]:
# recommendation function
title_to_idx = pd.Series(movies_pd.index, index=movies_pd["title"]).drop_duplicates()

def recommend_similar_movies(movie_title, top_n=5):
    if movie_title not in title_to_idx:
        return pd.DataFrame(columns=["movieId", "title", "genres_clean", "year", "similarity"])

    idx = title_to_idx[movie_title]
    sims = cosine_similarity(tfidf_matrix[idx], tfidf_matrix).flatten()
    similar_idx = np.argsort(-sims)
    similar_idx = [i for i in similar_idx if i != idx][:top_n]

    result = movies_pd.iloc[similar_idx][["movieId", "title", "genres_clean", "year"]].copy()
    result["similarity"] = sims[similar_idx]
    return result.reset_index(drop=True)

for title in ["Toy Story (1995)", "Jumanji (1995)", "Matrix, The (1999)"]:
    print(f"\nRecommendations for: {title}")
    display(recommend_similar_movies(title, 5))


Recommendations for: Toy Story (1995)


,movieId,title,genres_clean,year,similarity
0,45074,"Wild, The (2006)",Adventure|Animation|Children|Comedy|Fantasy,2006.0,1.0
1,3114,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy,1999.0,1.0
2,103755,Turbo (2013),Adventure|Animation|Children|Comedy|Fantasy,2013.0,1.0
3,166461,Moana (2016),Adventure|Animation|Children|Comedy|Fantasy,2016.0,1.0
4,65577,"Tale of Despereaux, The (2008)",Adventure|Animation|Children|Comedy|Fantasy,2008.0,1.0



Recommendations for: Jumanji (1995)


,movieId,title,genres_clean,year,similarity
0,126,"NeverEnding Story III, The (1994)",Adventure|Children|Fantasy,1994.0,1.0
1,119655,Seventh Son (2014),Adventure|Children|Fantasy,2014.0,1.0
2,4896,Harry Potter and the Sorcerer's Stone (a.k.a. ...,Adventure|Children|Fantasy,2001.0,1.0
3,80748,Alice in Wonderland (1933),Adventure|Children|Fantasy,1933.0,1.0
4,82169,Chronicles of Narnia: The Voyage of the Dawn T...,Adventure|Children|Fantasy,2010.0,1.0



Recommendations for: Matrix, The (1999)


,movieId,title,genres_clean,year,similarity
0,379,Timecop (1994),Action|Sci-Fi|Thriller,1994.0,1.0
1,71254,Gamer (2009),Action|Sci-Fi|Thriller,2009.0,1.0
2,3981,Red Planet (2000),Action|Sci-Fi|Thriller,2000.0,1.0
3,140956,Ready Player One,Action|Sci-Fi|Thriller,NaN,1.0
4,4443,Outland (1981),Action|Sci-Fi|Thriller,1981.0,1.0


## Task 2

In [10]:
# user-profile content recommender
train_pd = train_ratings.select("userId", "movieId", "rating").toPandas()
movieid_to_idx = pd.Series(movies_pd.index, index=movies_pd["movieId"]).drop_duplicates()

train_pd = train_pd[train_pd["movieId"].isin(movieid_to_idx.index)].copy()
train_pd["movie_idx"] = train_pd["movieId"].map(movieid_to_idx)

def build_user_profile(user_id):
    user_hist = train_pd[train_pd["userId"] == user_id]
    if user_hist.empty:
        return None

    idxs = user_hist["movie_idx"].values
    weights = user_hist["rating"].values.reshape(-1, 1)
    movie_vecs = tfidf_matrix[idxs].toarray()
    profile = (movie_vecs * weights).sum(axis=0) / weights.sum()
    return profile

def recommend_for_user_content(user_id, top_n=5):
    profile = build_user_profile(user_id)
    if profile is None:
        return pd.DataFrame(columns=["movieId", "title", "genres_clean", "score"])

    seen_movies = set(train_pd.loc[train_pd["userId"] == user_id, "movieId"])
    scores = cosine_similarity(profile.reshape(1, -1), tfidf_matrix).flatten()

    candidates = movies_pd.copy()
    candidates["score"] = scores
    candidates = candidates[~candidates["movieId"].isin(seen_movies)]
    return candidates.sort_values("score", ascending=False).head(top_n)[["movieId", "title", "genres_clean", "score"]]

active_users = train_pd.groupby("userId").size().reset_index(name="cnt")
sample_users = active_users[active_users["cnt"] >= 20]["userId"].head(3).tolist()

for uid in sample_users:
    print(f"\nUser-profile content recommendations for user {uid}")
    display(recommend_for_user_content(uid, 5))


User-profile content recommendations for user 1


,movieId,title,genres_clean,score
8597,117646,Dragonheart 2: A New Beginning (2000),Action|Adventure|Comedy|Drama|Fantasy|Thriller,0.803160
6570,55116,"Hunting Party, The (2007)",Action|Adventure|Comedy|Drama|Thriller,0.789474
4005,5657,Flashback (1990),Action|Adventure|Comedy|Crime|Drama,0.786155
4681,6990,The Great Train Robbery (1978),Action|Adventure|Comedy|Crime|Drama,0.786155
9394,164226,Maximum Ride (2016),Action|Adventure|Comedy|Fantasy|Sci-Fi|Thriller,0.775360



User-profile content recommendations for user 2


,movieId,title,genres_clean,score
118,145,Bad Boys (1995),Action|Comedy|Crime|Drama|Thriller,0.880794
1103,1432,Metro (1997),Action|Comedy|Crime|Drama|Thriller,0.880794
19,20,Money Train (1995),Action|Comedy|Crime|Drama|Thriller,0.880794
4693,7007,"Last Boy Scout, The (1991)",Action|Comedy|Crime|Drama|Thriller,0.880794
3989,5628,Wasabi (2001),Action|Comedy|Crime|Drama|Thriller,0.880794



User-profile content recommendations for user 3


,movieId,title,genres_clean,score
6789,60471,Rogue (2007),Action|Adventure|Horror|Sci-Fi|Thriller,0.926847
902,1200,Aliens (1986),Action|Adventure|Horror|Sci-Fi,0.910638
9361,161918,Sharknado 4: The 4th Awakens (2016),Action|Adventure|Horror|Sci-Fi,0.910638
6024,38583,"Wraith, The (1986)",Action|Horror|Sci-Fi|Thriller,0.897908
1009,1320,Alien³ (a.k.a. Alien 3) (1992),Action|Horror|Sci-Fi|Thriller,0.897908


In [11]:
# evaluate content recommender
test_pd = test_ratings.select("userId", "movieId", "rating").toPandas()

pred_rows = []
eval_users = sample_users[:]

for uid in eval_users:
    recs = recommend_for_user_content(uid, top_n=50)
    recs["userId"] = uid
    pred_rows.append(recs[["userId", "movieId", "score"]])

content_pred_df = pd.concat(pred_rows, ignore_index=True) if pred_rows else pd.DataFrame(columns=["userId", "movieId", "score"])
content_truth_df = test_pd[test_pd["userId"].isin(eval_users)].copy()

content_p5, content_r5 = precision_recall_at_k(content_pred_df, content_truth_df, k=5, positive_threshold=POSITIVE_THRESHOLD)
print("Content Precision@5:", round(content_p5, 4))
print("Content Recall@5   :", round(content_r5, 4))

Content Precision@5: 0.0
Content Recall@5   : 0.0


# Part 2 — Collaborative Filtering

In [12]:
# Filter active users and popular movies for fast CF
user_counts = train_ratings.groupBy("userId").count().filter(F.col("count") >= 30)
movie_counts = train_ratings.groupBy("movieId").count().filter(F.col("count") >= 50)

cf_train = train_ratings.join(user_counts.select("userId"), "userId") \
                        .join(movie_counts.select("movieId"), "movieId") \
                        .persist(StorageLevel.MEMORY_AND_DISK)

cf_pd = cf_train.select("userId", "movieId", "rating").toPandas()

user_item = cf_pd.pivot_table(index="userId", columns="movieId", values="rating")
user_item_filled = user_item.fillna(0)

print("User-item matrix shape:", user_item_filled.shape)

User-item matrix shape: (419, 339)


## Task 3

In [13]:
# User-based CF
user_similarity = cosine_similarity(user_item_filled.values)
user_similarity_df = pd.DataFrame(user_similarity, index=user_item_filled.index, columns=user_item_filled.index)

def predict_user_cf(user_id, movie_id, k=20):
    if user_id not in user_item_filled.index or movie_id not in user_item_filled.columns:
        return np.nan

    sims = user_similarity_df.loc[user_id].drop(user_id, errors="ignore")
    movie_ratings = user_item[movie_id].dropna()

    common_users = sims.index.intersection(movie_ratings.index)
    if len(common_users) == 0:
        return np.nan

    sims = sims.loc[common_users]
    movie_ratings = movie_ratings.loc[common_users]

    top_neighbors = sims.sort_values(ascending=False).head(k)
    neighbor_ratings = movie_ratings.loc[top_neighbors.index]

    denom = np.abs(top_neighbors).sum()
    if denom == 0:
        return np.nan

    return float((top_neighbors * neighbor_ratings).sum() / denom)

def recommend_user_cf(user_id, k=20, top_n=5):
    seen = set(cf_pd.loc[cf_pd["userId"] == user_id, "movieId"])
    candidates = [m for m in user_item_filled.columns if m not in seen]

    preds = []
    for m in candidates[:5000]:
        pred = predict_user_cf(user_id, m, k=k)
        if not np.isnan(pred):
            preds.append((m, pred))

    recs = pd.DataFrame(preds, columns=["movieId", "pred_rating"]).sort_values("pred_rating", ascending=False).head(top_n)
    return recs.merge(movies_pd[["movieId", "title", "genres_clean"]], on="movieId", how="left")

sample_user_cf = int(user_item_filled.index[0])
display(recommend_user_cf(sample_user_cf, k=20, top_n=5))

,movieId,pred_rating,title,genres_clean
0,858,4.704211,"Godfather, The (1972)",Crime|Drama
1,79132,4.543550,Inception (2010),Action|Crime|Drama|Mystery|Sci-Fi|Thriller|IMAX
2,750,4.540471,Dr. Strangelove or: How I Learned to Stop Worr...,Comedy|War
3,1221,4.530122,"Godfather: Part II, The (1974)",Crime|Drama
4,912,4.471188,Casablanca (1942),Drama|Romance


In [14]:
# evaluate user-based CF
cf_test_pd = test_ratings.select("userId", "movieId", "rating").toPandas()
cf_test_pd = cf_test_pd[
    cf_test_pd["userId"].isin(user_item_filled.index) &
    cf_test_pd["movieId"].isin(user_item_filled.columns)
].copy()

sample_eval = cf_test_pd.sample(min(2000, len(cf_test_pd)), random_state=42)

def eval_user_cf_rmse(k=20):
    y_true, y_pred = [], []
    for row in sample_eval.itertuples(index=False):
        pred = predict_user_cf(row.userId, row.movieId, k=k)
        if not np.isnan(pred):
            y_true.append(row.rating)
            y_pred.append(pred)
    return rmse_score(y_true, clip_ratings(np.array(y_pred))) if y_true else np.nan

for k in [10, 20, 50]:
    print(f"User-CF RMSE @ k={k}: {eval_user_cf_rmse(k):.4f}")

User-CF RMSE @ k=10: 0.7335
User-CF RMSE @ k=20: 0.7136
User-CF RMSE @ k=50: 0.6984


## Task 4

In [15]:
# Item-based CF
item_user = user_item.T
item_user_filled = item_user.fillna(0)

item_similarity = cosine_similarity(item_user_filled.values)
item_similarity_df = pd.DataFrame(item_similarity, index=item_user_filled.index, columns=item_user_filled.index)

def predict_item_cf(user_id, movie_id, k=20):
    if user_id not in user_item.index or movie_id not in item_similarity_df.index:
        return np.nan

    user_rated = user_item.loc[user_id].dropna()
    if user_rated.empty:
        return np.nan

    sims = item_similarity_df.loc[movie_id, user_rated.index]
    top_items = sims.sort_values(ascending=False).head(k)
    ratings_vec = user_rated.loc[top_items.index]

    denom = np.abs(top_items).sum()
    if denom == 0:
        return np.nan

    return float((top_items * ratings_vec).sum() / denom)

def recommend_item_cf(user_id, k=20, top_n=5):
    seen = set(cf_pd.loc[cf_pd["userId"] == user_id, "movieId"])
    candidates = [m for m in item_user.index if m not in seen]

    preds = []
    for m in candidates[:5000]:
        pred = predict_item_cf(user_id, m, k=k)
        if not np.isnan(pred):
            preds.append((m, pred))

    recs = pd.DataFrame(preds, columns=["movieId", "pred_rating"]).sort_values("pred_rating", ascending=False).head(top_n)
    return recs.merge(movies_pd[["movieId", "title", "genres_clean"]], on="movieId", how="left")

display(recommend_item_cf(sample_user_cf, k=20, top_n=5))

,movieId,pred_rating,title,genres_clean
0,32587,4.852224,Sin City (2005),Action|Crime|Film-Noir|Mystery|Thriller
1,58559,4.849649,"Dark Knight, The (2008)",Action|Crime|Drama|IMAX
2,1201,4.844523,"Good, the Bad and the Ugly, The (Buono, il bru...",Action|Adventure|Western
3,44191,4.818338,V for Vendetta (2006),Action|Sci-Fi|Thriller|IMAX
4,45722,4.814972,Pirates of the Caribbean: Dead Man's Chest (2006),Action|Adventure|Fantasy


In [16]:
# evaluate item-based CF
def eval_item_cf_rmse(k=20):
    y_true, y_pred = [], []
    for row in sample_eval.itertuples(index=False):
        pred = predict_item_cf(row.userId, row.movieId, k=k)
        if not np.isnan(pred):
            y_true.append(row.rating)
            y_pred.append(pred)
    return rmse_score(y_true, clip_ratings(np.array(y_pred))) if y_true else np.nan

for k in [10, 20, 50]:
    print(f"Item-CF RMSE @ k={k}: {eval_item_cf_rmse(k):.4f}")

Item-CF RMSE @ k=10: 0.7745
Item-CF RMSE @ k=20: 0.7769
Item-CF RMSE @ k=50: 0.7796


# Part 3 — Matrix Factorization

## Task 5

In [18]:
# Spark ALS
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator

als_train, als_test = ratings.randomSplit([0.8, 0.2], seed=42)

als = ALS(
    userCol="userId",
    itemCol="movieId",
    ratingCol="rating",
    rank=20,
    maxIter=10,
    regParam=0.1,
    coldStartStrategy="drop",
    nonnegative=True
)

als_model = als.fit(als_train)
als_predictions = als_model.transform(als_test)

reg_eval = RegressionEvaluator(metricName="rmse", labelCol="rating", predictionCol="prediction")
als_rmse = reg_eval.evaluate(als_predictions)

print("ALS RMSE:", round(als_rmse, 4))
als_predictions.show(5, truncate=False)

26/03/15 08:43:25 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS


ALS RMSE: 0.873
+------+-------+------+----------+----------+
|userId|movieId|rating|timestamp |prediction|
+------+-------+------+----------+----------+
|12    |357    |3.5   |1247264106|4.5203266 |
|12    |543    |3.5   |1247263318|4.0421023 |
|12    |830    |4.0   |1247263452|3.5002718 |
|12    |2072   |5.0   |1247263670|3.8089356 |
|12    |2717   |5.0   |1247263428|3.3206398 |
+------+-------+------+----------+----------+
only showing top 5 rows


In [19]:
# ALS top-5 recommendations
als_recs = als_model.recommendForAllUsers(5)
als_recs.show(3, truncate=False)

+------+-------------------------------------------------------------------------------------------------+
|userId|recommendations                                                                                  |
+------+-------------------------------------------------------------------------------------------------+
|1     |[{3030, 5.4985924}, {720, 5.4138594}, {177593, 5.407608}, {5075, 5.366146}, {3925, 5.3578453}]   |
|2     |[{131724, 4.8959217}, {2936, 4.6377263}, {86377, 4.5931125}, {78836, 4.5622635}, {3819, 4.55124}]|
|3     |[{6835, 4.9298697}, {5746, 4.9298697}, {5181, 4.8661604}, {4518, 4.747978}, {2851, 4.695036}]    |
+------+-------------------------------------------------------------------------------------------------+
only showing top 3 rows


In [20]:
# ALS Precision@5 and Recall@5
als_recs_exploded = als_recs.select(
    "userId",
    F.explode("recommendations").alias("rec")
).select(
    "userId",
    F.col("rec.movieId").alias("movieId"),
    F.col("rec.rating").alias("score")
)

als_pred_pd = als_recs_exploded.toPandas()
als_test_pd = als_test.select("userId", "movieId", "rating").toPandas()

als_p5, als_r5 = precision_recall_at_k(als_pred_pd, als_test_pd, k=5, positive_threshold=POSITIVE_THRESHOLD)
print("ALS Precision@5:", round(als_p5, 4))
print("ALS Recall@5   :", round(als_r5, 4))

ALS Precision@5: 0.003
ALS Recall@5   : 0.0009


In [21]:
# Local SVD on filtered matrix
R = user_item.fillna(0).values
user_ids_local = user_item.index.to_list()
movie_ids_local = user_item.columns.to_list()

k_latent = min(20, min(R.shape) - 1)
U, s, Vt = svds(R, k=k_latent)
sigma = np.diag(s)
R_hat = np.dot(np.dot(U, sigma), Vt)
R_hat = clip_ratings(R_hat)

R_hat_df = pd.DataFrame(R_hat, index=user_ids_local, columns=movie_ids_local)
print("Local SVD reconstructed matrix shape:", R_hat_df.shape)

Local SVD reconstructed matrix shape: (419, 339)


## Task 6

In [22]:
# Surprise SVD
try:
    from surprise import Dataset, Reader, SVD
    from surprise.model_selection import train_test_split as surprise_train_test_split
    from surprise import accuracy

    surprise_df = train_ratings.select("userId", "movieId", "rating").limit(500000).toPandas()
    reader = Reader(rating_scale=(0.5, 5.0))
    surprise_data = Dataset.load_from_df(surprise_df[["userId", "movieId", "rating"]], reader)

    trainset, testset = surprise_train_test_split(surprise_data, test_size=0.2, random_state=42)
    surprise_algo = SVD(n_factors=50, n_epochs=20, random_state=42)
    surprise_algo.fit(trainset)
    surprise_preds = surprise_algo.test(testset)

    surprise_rmse = accuracy.rmse(surprise_preds, verbose=False)
    print("Surprise SVD RMSE:", round(surprise_rmse, 4))
except Exception as e:
    print("Surprise section skipped:", e)

Surprise section skipped: numpy.core.multiarray failed to import (auto-generated because you didn't call 'numpy.import_array()' after cimporting numpy; use '<void>numpy._import_array' to disable if you are certain you don't need it).



A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.3 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/Users/ashishsinha/Documents/Study/MTech/Sm02/MLwDE/Assignment03/gihubrepo/iitj_m25de1047_mlbd_Assignment03/venv/lib/python3.12/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/Users/ashishsinha/Documents/Study/MTech/Sm02/MLwDE/Assignment03/gihubrepo/iitj_m25de1047_mlbd_Assignment03/venv/lib/python3.12/site-packages/traitlets/config/application.py", line 1075, in 